# CS383: Data Science and Machine Learning
## Lecture 2 — Python Refresher, NumPy, and Vectorized Computing

*Dr. Thitima Srivatanakul*

### Guiding question
**Why do data scientists reach for NumPy arrays instead of plain Python loops?**

### Learning objectives
By the end of this lecture, you should be able to:

- refresh core Python skills (variables, conditionals, lists, dictionaries, loops, functions, and basic error handling) in a data-analysis context;
- create, index, and slice NumPy arrays;
- explain what "vectorized" computation means, in your own words;
- rewrite a loop-based calculation as an equivalent vectorized NumPy operation;
- measure and compare the running time of a loop vs. a vectorized version of the same calculation;
- use vectorized operations to compute basic summary statistics on a real dataset.

---

### Where this fits
In Lecture 1 you got a first taste of pulling and charting real NYC 311 data. This lecture builds the coding foundation that made that possible: Python fundamentals and NumPy's vectorized way of computing. Pandas and SQL — the tools you'll actually use to wrangle data like the 311 dataset day to day — start in Lecture 3.

---
**Live in-class version.** Type along at each `__________` blank — everything else is filled in so class time stays on the new syntax, not on retyping boilerplate.

---

### Before we start: a 60-second preview

Everything in this lecture builds toward one idea: **the same calculation can be written two different ways in Python, and one of those ways can be 50–100x faster.** Run the cell below and watch the clock.

In [ ]:
import numpy as np
import time

n = 5_000_000
values = np.random.default_rng(383).random(n)

# Version 1: plain Python loop
start = time.time()
total_loop = 0.0
for v in values:
    total_loop += v ** 2
loop_seconds = time.time() - start

# Version 2: vectorized NumPy
start = time.time()
total_vectorized = np.sum(values ** 2)
vectorized_seconds = time.time() - start

print(f"Loop version:       {loop_seconds:.3f} seconds")
print(f"Vectorized version: {vectorized_seconds:.5f} seconds")
print(f"Vectorized was about {loop_seconds / vectorized_seconds:,.0f}x faster")
print(f"Same answer? {np.isclose(total_loop, total_vectorized)}")

### What just happened?

Both versions computed the exact same thing: the sum of 5 million squared numbers.

- The **loop version** asks Python to check, interpret, and execute one instruction at a time — 5 million separate times.
- The **vectorized version** hands the whole array to NumPy, which does the same math in fast, pre-compiled C code, operating on the entire array in one call.

By the end of this lecture, you'll be able to write code like the vectorized version yourself — and explain why it's faster.

---

## Part 1 — Python Refresher

A quick review before we add NumPy on top. If any of this feels unfamiliar, flag it now — it's foundational for the rest of the course.

In [ ]:
# Variables and basic types
n_complaints = 1523        # int
avg_response_hours = 4.75  # float
borough = "BROOKLYN"       # str
is_resolved = __________   # bool

print(type(n_complaints), type(avg_response_hours), type(borough), type(is_resolved))

### f-strings: formatting output

In [ ]:
n_complaints = 1523
avg_response_hours = 4.75

print(f"There were {n_complaints} complaints, averaging {avg_response_hours} hours to resolve.")

# Format specs (after the colon) control how numbers are displayed
print(f"Average response time: {avg_response_hours:__________}")   # 1 decimal place
print(f"Total complaints: {n_complaints:,}")                      # comma thousands separator
print(f"{'Borough':15s} {'Count':>10s}")                          # column widths/alignment

Everywhere you see `f"...{something:...}..."` in this course's code, that's an f-string: `.1f` means "1 decimal place," `,` means "add thousands separators," `15s` means "pad this text to 15 characters wide." You'll see these same formatting patterns throughout both labs later in this lecture.

### Conditionals

In [ ]:
hours_open = 6

if hours_open < 2:
    priority = "low"
elif hours_open < __________:
    priority = "medium"
else:
    priority = "high"

print(priority)

In [ ]:
borough = "BROOKLYN"
is_urgent = True

if borough == "BROOKLYN" and is_urgent:
    print("Dispatch immediately")
elif borough == "BROOKLYN" __________ is_urgent:
    print("Flag for review")
else:
    print("Standard queue")

In [ ]:
# Lists
complaint_counts = [1523, 987, 2210, 640, 88]
boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

print(complaint_counts[0])       # first element
print(complaint_counts[-1])      # last element
print(complaint_counts[__________])     # slice
print(len(complaint_counts))     # length

### Dictionaries

In [ ]:
complaint_record = {
    "complaint_type": "Noise - Residential",
    "borough": "BROOKLYN",
    "created_date": "2026-08-10T14:32:00.000",
}

print(complaint_record["complaint_type"])         # access by key
print(complaint_record.get("borough"))            # safer access
print(complaint_record.get("status", "__________"))  # default value if the key is missing

for key, value in complaint_record.items():
    print(f"{key}: {value}")

This `.get(key, default)` pattern is exactly what shows up when we pull real 311 records in this lecture's exercises — each record arrives from the API as a dictionary, and `.get(...)` protects the code from crashing if a field happens to be missing.

In [ ]:
# For loops
total = 0
for count in complaint_counts:
    total __________ count
print("Total complaints:", total)

In [ ]:
# List comprehensions
doubled = [count * 2 for count in complaint_counts]
print(doubled)

high_volume = [b for b, c in zip(boroughs, complaint_counts) if c __________ 1000]
print(high_volume)

In [ ]:
# Functions
def average(values):
    return sum(values) / __________(values)

print(average(complaint_counts))

### try / except: handling things that might fail

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except __________:
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

In [ ]:
try:
    result = 10 / 0
    print("Success:", result)
except Exception as e:
    print(f"Something went wrong ({type(e).__name__}), using a fallback value instead.")
    result = __________

print(result)

This is exactly the pattern behind the live NYC 311 pulls you've seen since Lecture 1: `try` the real API call, and if anything goes wrong — no internet, a slow server, a changed API — `except` catches it and falls back to offline sample data instead of crashing the whole notebook.

### Quick check

Predict the output of each line below, then run the cell to check yourself.

In [ ]:
print(complaint_counts[2:])
print([c for c in complaint_counts if c < 1000])
print(average(complaint_counts[:2]))
print(complaint_record.get("priority", "not set"))

---

## Part 2 — Introducing NumPy Arrays

Lists are flexible, but they're slow for numeric work and don't support math the way you'd expect.

In [ ]:
complaint_counts_list = [1523, 987, 2210, 640, 88]

# This does NOT do what you might expect:
print(complaint_counts_list * 2)

`list * 2` repeats the list's contents twice — it does not multiply each number by 2. Lists don't know how to do elementwise math. This is exactly the gap NumPy fills.

In [ ]:
import numpy as np

complaint_counts = np.__________([1523, 987, 2210, 640, 88])
print(complaint_counts)
print(type(complaint_counts))
print(complaint_counts * 2)   # this DOES multiply every element

### Array basics

In [ ]:
arr = np.array([10, 20, 30, 40, 50])

print(arr.shape)   # dimensions
print(arr.__________)   # data type
print(arr.ndim)    # number of dimensions
print(arr.size)    # total number of elements

### Creating arrays without typing every value

In [ ]:
zeros = np.zeros(5)
ones = np.ones(5)
sequence = np.__________(0, 50, 10)       # start, stop, step
evenly_spaced = np.linspace(0, 1, 5)  # start, stop, number of points

print(zeros)
print(ones)
print(sequence)
print(evenly_spaced)

### Indexing and slicing (same syntax as lists!)

In [ ]:
print(arr[0])
print(arr[-1])
print(arr[1:3])
print(arr[arr __________ 20])   # boolean indexing -- new!

### Boolean indexing

`arr[arr > 20]` reads as "give me the elements of `arr` where the condition is True." `arr > 20` on its own produces an array of `True`/`False` values — one per element — which NumPy then uses to filter.

In [ ]:
print(arr > 20)
print(arr[arr __________ 20])

### 2D arrays (a preview)

In [ ]:
grid = np.array([[1, 2, 3], [4, 5, 6]])
print(grid.shape)
print(grid[0])      # first row
print(grid[:, __________])   # second column

---

## Part 3 — Vectorized Computation

**Vectorization** means applying an operation to an entire array at once, with no explicit loop written by you.

### Elementwise arithmetic

In [ ]:
temps_f = np.array([32, 50, 68, 86, 104])

temps_c_loop = []
for t in temps_f:
    temps_c_loop.append((t - 32) * 5 / 9)

temps_c_vectorized = (temps_f - 32) * __________ / 9

print(temps_c_loop)
print(temps_c_vectorized)

Same result, two different styles. From here on, we'll almost always prefer the vectorized style.

### Aggregate functions

In [ ]:
print(temps_c_vectorized.sum())
print(temps_c_vectorized.mean())
print(temps_c_vectorized.max())
print(temps_c_vectorized.min())
print(temps_c_vectorized.__________())

### Broadcasting

When you do math between an array and a single number — or between two differently-shaped arrays that are compatible — NumPy "stretches" the smaller one to match. That's called **broadcasting**, and it's why `(temps_f - 32) * 5 / 9` above worked without a loop, even though `32`, `5`, and `9` are plain numbers, not arrays.

In [ ]:
prices = np.array([100, 250, 40, 15])
tax_rate = 0.08

total_prices = prices * (1 + __________)
print(total_prices)

---

## Part 4 — Why Vectorized Code Is Faster

- Python loops process one element at a time, and each step of the interpreter carries real overhead.
- NumPy vectorized operations run in pre-compiled C code that processes the whole array in a single call, without that per-element interpreter cost.
- This matters more as data gets bigger. Remember the 60-second preview? At 5 million elements, the difference was dramatic. At 5 elements, you'd barely notice.
- Vectorization isn't a trick or shortcut — it's the normal, idiomatic way to write numeric Python code. Pandas, scikit-learn, and virtually every other data science library are built on top of NumPy arrays for exactly this reason.

### Discussion
- Look back at the 60-second preview. At roughly what array size do you think the speed difference would become invisible to a human?
- Can you think of a calculation where a loop might still be necessary, even in a data science workflow?

### A convenience that isn't what it looks like: `np.vectorize()`

NumPy has a function literally named `np.vectorize()` — which sounds like exactly what you want. It's a trap. It doesn't run in fast compiled code; it just loops through your array in plain Python and calls your function once per element, dressed up to *look* like a single vectorized call. It's convenience, not speed.

In [ ]:
def celsius_to_fahrenheit(c):
    return c * 9 / 5 + 32

values = np.random.default_rng(383).random(2_000_000) * 100
fake_vectorized = np.vectorize(celsius_to_fahrenheit)

start = time.time()
result_fake = fake_vectorized(values)
fake_time = time.time() - start

start = time.time()
result_real = values * 9 / 5 + __________
real_time = time.time() - start

print(f"np.vectorize():      {fake_time:.3f} seconds")
print(f"Real vectorization:  {real_time:.5f} seconds")
print(f"Real vectorization was about {fake_time / real_time:,.0f}x faster")
print(f"Same answer? {np.allclose(result_fake, result_real)}")

Despite the name, `np.vectorize()` is really just a `for` loop wearing a NumPy costume — it exists to let you reuse a function you already wrote without rewriting it as array operations, not to make it fast. If a genuinely vectorized version exists (plain arithmetic, like above), prefer that. You'll meet this same disguised-loop trap again in Lecture 3, in the form of Pandas' `.apply()`.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook: `lect02_python_numpy_vectorization_exercise.ipynb`.

---

## Part 5 — Key Terms

- **Array**: a NumPy data structure holding elements of a single type, supporting fast elementwise math.
- **Vectorization**: applying an operation to an entire array at once instead of looping element by element.
- **Elementwise operation**: an operation applied independently to each element (e.g., `arr + 1`).
- **Broadcasting**: NumPy's rule for applying operations between arrays (or an array and a scalar) of compatible shapes, without writing a loop.
- **Boolean indexing / mask**: selecting elements of an array using a `True`/`False` condition (e.g., `arr[arr > 20]`).
- **Aggregate function**: a function that reduces an array to a single summary value (`sum`, `mean`, `max`, `std`, ...).
- **shape / dtype / ndim / size**: NumPy array attributes describing an array's dimensions, element type, number of dimensions, and total element count.